In [ ]:
# Example 1: is_now() against the current time should return a boolean.
import pycron

# '* * * * *' matches every minute, so is_now should always be True.
result_every = pycron.is_now('* * * * *')
print(f"is_now('* * * * *') = {result_every}")
assert result_every is True

# A clearly bogus minute (61) is not a valid cron field; verify behavior on a
# legitimate but unlikely-to-match expression instead.
result_specific = pycron.is_now('0 0 1 1 *')  # Jan 1 at midnight
print(f"is_now('0 0 1 1 *') = {result_specific}")
assert isinstance(result_specific, bool)


In [ ]:
# Example 2: is_now() with an explicit datetime — deterministic checks.
import datetime

import pycron

dt = datetime.datetime(2024, 1, 1, 0, 0)

assert pycron.is_now('0 0 1 1 *', dt) is True
assert pycron.is_now('0 0 * * *', dt) is True       # any midnight
assert pycron.is_now('30 0 * * *', dt) is False     # 00:30 only
assert pycron.is_now('* * * * 1', dt) is True       # Monday (cron: 1=Mon)

print("is_now() deterministic checks passed for 2024-01-01 00:00")

In [ ]:
# Example 3: step values, ranges, and lists in cron fields.
import datetime

import pycron

dt_15 = datetime.datetime(2024, 6, 15, 12, 15)
dt_20 = datetime.datetime(2024, 6, 15, 12, 20)

# */5 — every 5 minutes
assert pycron.is_now('*/5 * * * *', dt_15) is True
assert pycron.is_now('*/5 * * * *', dt_20) is True

# Range: minutes 10-20
assert pycron.is_now('10-20 * * * *', dt_15) is True

# List: specific minutes
assert pycron.is_now('15,45 * * * *', dt_15) is True
assert pycron.is_now('15,45 * * * *', dt_20) is False

print("Step / range / list cron expressions evaluated correctly")


In [ ]:
# Example 4: has_been() — was this cron schedule due in a recent window?
import datetime

import pycron

# Reference time well after midnight.
now = datetime.datetime(2024, 1, 1, 12, 0)

# Cron firing daily at midnight — over the last 24h, it should have fired.
since_24h_ago = now - datetime.timedelta(hours=24)
assert pycron.has_been('0 0 * * *', since_24h_ago, now) is True

# A schedule that fires only on Feb 29 — should not have fired in this window.
assert pycron.has_been('0 0 29 2 *', since_24h_ago, now) is False

print("has_been() correctly reports past schedule firings")


In [ ]:
# Example 5: invalid cron expressions should raise (negative-path coverage).
import pycron

bad_expressions = [
    'not a cron expression',
    '* * *',                   # too few fields
    '* * * * * * *',           # too many fields
]

raised = 0
for expr in bad_expressions:
    try:
        pycron.is_now(expr)
    except ValueError as e:
        raised += 1
        print(f"  {expr!r} -> {type(e).__name__}")
    else:
        raise AssertionError(f"pycron accepted the malformed expression {expr!r}")

# The structural ones (wrong field count, non-cron text) must all fail.
assert raised == len(bad_expressions), f"Expected every malformed expression to raise, got {raised}"

# pycron does not range-check a field: an out-of-range minute parses and simply never matches
assert pycron.is_now('99 * * * *') is False
print(f"\n{raised}/{len(bad_expressions)} invalid cron expressions rejected")